# 01 — GeoChat: VQA + Captioning + Remote-Sensing Adaptation
**Owner: Person 1**

Priority 1 (mandatory): remote-sensing adaptation + single-image VQA.
This notebook also covers the team's chosen 2nd single-image task (captioning), sharing the same GeoChat checkpoint.

**Do not fine-tune a generic BLIP-2/LLaVA** — GeoChat is the team's chosen backbone for this slot; it is already remote-sensing-adapted, and we further adapt it with LoRA on a BigEarthNet.txt subset to make the adaptation requirement defensible.

Workflow: BigEarthNet.txt subset -> image/text pairs -> pretrained GeoChat -> LoRA fine-tuning -> checkpoint -> VQA eval -> caption eval.

## 1. Environment

In [ ]:
# !pip install torch transformers peft accelerate rasterio pillow --quiet
import sys, os
sys.path.insert(0, os.path.abspath('..'))


## 2. Load shared configuration
Everyone reads from the same `configs/config.yaml` — do not hardcode paths/params here.

In [ ]:
from src.utils.io_utils import load_config

config = load_config('../configs/config.yaml')
vqa_cfg = config['models']['vqa']
caption_cfg = config['models']['captioning']
bigearthnet_cfg = config['datasets']['bigearthnet']
vqa_cfg, bigearthnet_cfg

## 3. Inspect BigEarthNet.txt
Confirm the actual label-file format before writing the parser in `src/preprocessing/bigearthnet_adapter.py`.

In [ ]:
# labels_path = bigearthnet_cfg['labels_file']
# with open(labels_path) as f:
#     sample_lines = [next(f) for _ in range(5)]
# sample_lines


## 4. Build image-text adaptation pairs
Uses `src/preprocessing/bigearthnet_adapter.py` once implemented.

In [ ]:
from src.preprocessing.bigearthnet_adapter import build_pretraining_pairs

# pairs = build_pretraining_pairs(
#     labels_file=bigearthnet_cfg['labels_file'],
#     image_dir=bigearthnet_cfg['image_dir'],
# )
# pairs[:3]


## 5. Load pretrained GeoChat (baseline, before adaptation)
Run this BEFORE any fine-tuning so you can show a before/after comparison in your final report.

In [ ]:
# from transformers import AutoModel, AutoProcessor
# geochat = AutoModel.from_pretrained('<geochat-checkpoint>')
# processor = AutoProcessor.from_pretrained('<geochat-checkpoint>')


## 6. Baseline VQA inference (pretrained, no adaptation)

In [ ]:
# sample_image = '<path to a sample RS image>'
# sample_question = 'What type of land cover is visible in this image?'
# baseline_answer = ...  # run geochat inference here
# baseline_answer


## 7. Configure LoRA adaptation
See `src/models/vqa_model.py` — this is where the reusable version of this cell eventually lives.

In [ ]:
# from peft import LoraConfig, get_peft_model
# lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=[...], lora_dropout=0.05)
# geochat_lora = get_peft_model(geochat, lora_config)


## 8. Fine-tune on BigEarthNet-adapted pairs (+ RSVQA for VQA task)

In [ ]:
# training loop here — keep it short for the sprint, checkpoint often
# for epoch in range(...):
#     for batch in train_loader:
#         loss = train_step(geochat_lora, batch, optimizer, config)


## 9. Save the LoRA adapter checkpoint
Save to the path declared in config: `models.vqa.checkpoint`.

In [ ]:
# geochat_lora.save_pretrained(vqa_cfg['checkpoint'])
# print('Saved to', vqa_cfg['checkpoint'])


## 10. Test the adapted model (after adaptation)
Compare directly against Section 6's baseline answer.

In [ ]:
# adapted_answer = ...  # run geochat_lora inference on the same sample
# adapted_answer


## 11. Evaluate on RSVQA / VRSBench
Use `src/evaluation/metrics.py::vqa_accuracy` and `caption_scores`.

In [ ]:
from src.evaluation import metrics as M

# preds, golds = [], []  # collect from eval loop
# print('VQA accuracy:', M.vqa_accuracy(preds, golds))


## 12. Export the inference function
Once this notebook works end-to-end, move the stable `predict()` implementation into `src/models/vqa_model.py` (and `src/models/captioning_model.py`) so the agent controller can call it via the shared `BaseRSModel` / `RSModelResult` contract — do not leave the only working copy inside this notebook.

In [ ]:
from src.common.schemas import RSModelResult, ResultMetadata, Evidence

# def predict(image, query):
#     answer = ...
#     confidence = ...
#     return RSModelResult(
#         task='vqa', answer=answer, confidence=confidence,
#         evidence=Evidence(),
#         metadata=ResultMetadata(model='GeoChat',
#                                  checkpoint=vqa_cfg['checkpoint'],
#                                  dataset='BigEarthNet.txt+RSVQA',
#                                  backbone='GeoChat'),
#     ).to_dict()
